# Basic RAG

- RAG는 LLM의 장점과 knowledge base에서 관련된 정보를 검색하는 능력을 결합한 technique.
- 이는 생성도니 응답을 특정 검색 정보에 기반해서 생성하므로, 응답의 품질과 정확성을 향상시켜 줌.


- 기존 LM들은 train data에서 학습한 패턴을 기반으로 text를 생성했음.
- 하지만 특정 정보, 최신 정보가 필요한 query가 주어지면 정확한 답변을 제공하는데 어려움을 겪을 수 있음.
- RAG는 LM에 관련된 context를 제공해주므로, **보다 정보에 기반한 답변을 생성할 수 있도록 검색 단계를 붙여서 한계점을 해결**함.


그 과정을 살펴보면...

### 1. Document preprocessing and Vector store creation

1. **Document chunking**: knowledge base의 문서(e.g. PDF, 논문)들은 전처리 과정을 거쳐 **관리하기 쉬운 chunk 단위로 분할**.
   - retrieval(검색) 과정에서 효율적으로 사용할 수 있는 searchable corpus를 생성하기 위함.
2. **Embedding generation**: 각 chunk는 **pre-trained embedding(e.g. OpenAI의 embedding)을 사용해 vector representation으로 변환**.
   - 이를 통해 문서를 **Qdrant와 같은 vector DB에 저장**해 효율적인 similarity search가 가능해짐.

### 2. Retrieval-Augmented Generation workflow

1. **Query input**: 사용자가 답변을 얻고자 하는 질문을 제시
2. **Retrieval step**: query는 앞서 embedding generation에서 사용했던 것과 동일한 embedding model을 사용해 vector로 변환됨.
   - 이후 **vector DB에서 similarity search를 수행해 가장 관련성이 높은 chunk를 탐색**.
3. **Generation step**: **검색된 문서 chunk들은 추가적인 context 정보로 LLM(e.g. GPT-4)에 전달**됨.
   - 모델은 이 context 정보를 활용해서 더욱 정확하고 관련성 높은 응답을 생성함.


## RAG의 주요 특징들

- **Contextual Relevance**: RAG 모델들은 실제 검색된 정보를 기반으로 응답을 생성하므로, **맥락적으로 더 관련성이 높고 정확한 답변을 제공**할 수 있게 됨.
- **Scalability**: retrieval 단계는 대규모의 knowledge base를 처리할 수 있도록 확장이 가능.
  - 즉, **모델이 방대한 양의 정보를 활용**할 수 있음.
- **Flexibility in Use Cases**: QA, 요약, 추천시스템 등 **다양한 application에서 활용 가능**함.
- **Improved Accuracy**: retrieval과 generation을 결합하면, **잘 알려지지 않은 정보가 필요한 query의 경우 더 정확한 결과**를 얻을 수 있음.

그 외에도...

- 검색 기반의 방법과 생성형 모델을 효과적으로 결합 $\rightarrow$ 정확한 사실 검색과 자연어 생성이 모두 가능
- 발생 빈도가 낮은(long-tail) 정보가 필요한 query에 효과적
- 검색 메커니즘을 특정 도메인에 맞게 조정이 가능
  - 생성된 응답이 해당 도메인에 가장 관련성이 높고 정확한 정보를 내놓도록 보장


## Basic Environment Setup

- example은 OpenAI와 Grok을 사용하지만, 여기선 비용 문제로 Google 사용함.
- [LlamaIndex 문서 참고 링크](https://developers.llamaindex.ai/python/framework/integrations/llm/google_genai/)


In [1]:
from rich import print # 이쁜 output
from importlib.metadata import version

packages = ["numpy",
            "scipy",
            "torch",
            "sentence-transformers",
            "wikipedia-api",
            "google-genai",
            "rich",
            "pypdf2",
            "gradio",
            "llama-index",
            "llama-index-vector-stores-qdrant",
            "llama-index-readers-file",
            "llama-index-embeddings-fastembed",
            "llama-index-llms-google-genai",
            "qdrant_client",
            "fastembed",
            "gradio"
            ]

for package in packages:
    try:
        pkg_version = version(package)
    except Exception as e:
        pkg_version = f"Error retrieving version: {e}"
    print(f"{package}: {pkg_version}")

numpy: 2.1.3

scipy: 1.14.1

torch: 2.4.1

sentence-transformers: 5.3.0

wikipedia-api: 0.14.0

...

llama-index-embeddings-fastembed: 0.6.0

llama-index-llms-google-genai: 0.9.1

qdrant_client: 1.17.1

fastembed: 0.8.0

gradio: 6.11.0

In [3]:
# Standard library
import logging
import sys
import os

# Third-party import
from IPython.display import Markdown, display

# Qdrant client
import qdrant_client

# LlamaIndedx core
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import Settings

# LlamaIndex vector store
from llama_index.vector_stores.qdrant import QdrantVectorStore

# Embedding model
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding

# LLM
from llama_index.llms.google_genai import GoogleGenAI

# API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Base LLM setting
Settings.llm = GoogleGenAI(
    model = 'gemini-2.5-flash',
    api_key = GEMINI_API_KEY
)

Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-base-en-v1.5")

## Data Loading


In [4]:
from llama_index.core import Document

reader = SimpleDirectoryReader(os.getcwd()+'/rag_dataset', recursive=True)
documents = reader.load_data(show_progress=True)

Loading files: 100%|██████████| 2/2 [00:01<00:00,  1.99it/s]


## Setting up Vector Database

- qDrant를 vector DB로 활용.


In [5]:
client = qdrant_client.QdrantClient(
    path = './rag_dataset'
)

vector_store = QdrantVectorStore(client=client, collection_name="01_Basic_RAG")

## Vector DB에 data 넣기


In [6]:
# Set up ingestion pipeline

# from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.node_parser import SentenceSplitter
# from llama_index.core.node_parser import MarkdownNodeParser
# from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.ingestion import IngestionPipeline

# 파이프라인 통해서 안에 있는 문서들을 chunk 단위로 분할
# 이때 embedding 하는 모델은 앞서 선언한 embed_model을 사용
pipeline = IngestionPipeline(
    transformations = [
        SentenceSplitter(chunk_size=1024, chunk_overlap=20),
        Settings.embed_model
    ],
    vector_store = vector_store
)

# vector DB에 chunk들을 넣기
nodes = pipeline.run(documents=documents, show_progress=True)
print("Number of chunks added to vector DB :",len(nodes))

Applying transformations: 100%|██████████| 2/2 [00:09<00:00,  4.58s/it]
c:\Users\skdbs\.conda\envs\torch\Lib\site-packages\llama_index\vector_stores\qdrant\base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(


Number of chunks added to vector DB : 12

## Index Setup


In [7]:
# 유사도 검색 indexing을 위한 준비
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

## Promtpt 수정 및 Tuning


In [8]:
from llama_index.core import ChatPromptTemplate

qa_prompt_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the question: {query_str}\n"
)

refine_prompt_str = (
    "We have the opportunity to refine the original answer "
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{context_msg}\n"
    "------------\n"
    "Given the new context, refine the original answer to better "
    "answer the question: {query_str}. "
    "If the context isn't useful, output the original answer again.\n"
    "Original Answer: {existing_answer}"
)

# Text QA Prompt
chat_text_qa_msgs = [
    ("system","You are a AI assistant who is well versed with answering questions from the provided context"),
    ("user", qa_prompt_str),
]
text_qa_template = ChatPromptTemplate.from_messages(chat_text_qa_msgs)

# Refine Prompt
chat_refine_msgs = [
    ("system","Always answer the question, even if the context isn't helpful.",),
    ("user", refine_prompt_str),
]
refine_template = ChatPromptTemplate.from_messages(chat_refine_msgs)

In [9]:
# asyncio error 나는 경우 이거 사용
import nest_asyncio
nest_asyncio.apply()

chat_query = "What is in this document"

# Chat engine setup
BASE_RAG_CHAT_ENGINE = index.as_chat_engine()

response = BASE_RAG_CHAT_ENGINE.chat(chat_query)
display(Markdown(str(response)))

2026-04-04 20:51:22,659 - INFO - Condensed question: What is in this document
2026-04-04 20:51:23,043 - INFO - AFC is enabled with max remote calls: 10.
2026-04-04 20:51:31,595 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


I have information from two different documents.

The first document, titled "**Mycophenolate therapy in interstitial pneumonia with autoimmune features: a cohort study**," is an article from the journal *Ther Clin Risk Manag*, published in 2018. It discusses:

*   **Objectives:** The study aimed to characterize patients with Interstitial Pneumonia with Autoimmune Features (IPAF) and evaluate the effectiveness of mycophenolate as a therapy for this condition. IPAF is described as a provisional diagnosis for patients with interstitial lung disease who show characteristics of autoimmune disease but don't meet the full criteria for a specific autoimmune disease.
*   **Methods:** It was a retrospective cohort study involving adult patients who met the European Respiratory Society/American Thoracic Society classification criteria for IPAF. Researchers collected sociodemographic, clinical, and pulmonary function test (PFT) data, following patients longitudinally to observe changes in PFT results, comparing those who received mycophenolate treatment with those who did not.
*   **Results:** The study identified 52 IPAF patients, with 24 not receiving mycophenolate and 28 who did (median time to treatment was 22 months). While there was no statistically significant difference in changes in FVC% (forced vital capacity percentage) and DLCO% (percentage predicted lung diffusion capacity for carbon monoxide) between the treated and untreated groups, a trend was observed. Specifically, the mycophenolate-treated cohort showed a more rapid decline in both FVC% and DLCO% *before* starting therapy. After mycophenolate exposure, the slope of both FVC% and DLCO% values improved for the treated group, although this improvement was not statistically significant.
*   **Conclusion:** The study suggests that patients with IPAF *might* benefit from mycophenolate therapy, but it emphasizes the need for larger prospective clinical trials to definitively evaluate its efficacy.
*   **Keywords:** The article is associated with keywords such as interstitial lung disease, autoimmune disease, connective tissue disease, and mycophenolate.
*   **Introduction:** It further clarifies IPAF as a research term for patients with an interstitial lung process consistent with idiopathic interstitial pneumonia (IP) combined with features of autoimmunity, without meeting full diagnostic criteria for a specific connective tissue disease (CTD). It also mentions that a patient must meet criteria from two of three prespecified domains to fulfill IPAF criteria: clinical features of extrathoracic autoimmune disease, serologic evidence of autoimmune disease, and morphological criteria. The document notes a lack of information on treatment recommendations or clinical outcomes for IPAF patients despite its clinical familiarity.

The second document is a **list of references**, likely from a research paper, possibly titled "Attention Is All You Need" given the file name. This list includes various academic papers related to machine learning, neural networks, and natural language processing. Some of the notable references include:

*   **Kyunghyun Cho et al. (2014):** "Learning phrase representations using rnn encoder-decoder for statistical machine translation."
*   **Francois Chollet (2016):** "Xception: Deep learning with depthwise separable convolutions."
*   **Junyoung Chung et al. (2014):** "Empirical evaluation of gated recurrent neural networks on sequence modeling."
*   **Jonas Gehring et al. (2017):** "Convolutional sequence to sequence learning."
*   **Alex Graves (2013):** "Generating sequences with recurrent neural networks."
*   **Kaiming He et al. (2016):** "Deep residual learning for image recognition."
*   **Sepp Hochreiter and Jürgen Schmidhuber (1997):** "Long short-term memory."
*   **Łukasz Kaiser and Samy Bengio (2016):** "Can active memory replace attention?"
*   **Diederik Kingma and Jimmy Ba (2015):** "Adam: A method for stochastic optimization."

These references cover topics such as RNNs, LSTMs, convolutional networks, deep learning architectures, machine translation, sequence modeling, attention mechanisms, and optimization algorithms.

## Simple Chat application with RAG


- 이 부분은 .py 파일 참고


In [13]:
from typing import List
from llama_index.core.base.llms.types import ChatMessage, MessageRole

class ChatEngineInterface:
    def __init__(self, index):
        self.chat_engine = index.as_chat_engine()
        self.chat_history: List[ChatMessage] = []

    def display_message(self, role:str, content:str):
        if role == "USER":
            display(Markdown(f"**Human:** {content}"))
        else:
            display(Markdown(f"**AI:** {content}"))
    
    def chat(self, message:str) -> str:
        # user input을 위한 ChatMessage 생성
        user_message = ChatMessage(role=MessageRole.USER, content=message)
        self.chat_history.append(user_message)

        # chat engine에서 response 받기
        response = self.chat_engine.chat(message, chat_history=self.chat_history)

        # AI 응답을 위한 ChatMessage 생성
        ai_message = ChatMessage(role=MessageRole.ASSISTANT, content=str(response))
        self.chat_history.append(ai_message)

        # 대화 내역 display
        self.display_message("USER", message)
        self.display_message("ASSITANT", str(response))

        return str(response)
    
    def get_chat_history(self) -> List[ChatMessage]:
        return self.chat_history

동작 예시를 보면...


In [ ]:
chat_interface = ChatEngineInterface(index)
while True:
    user_input = input("You: ").strip()

    if user_input.lower() == 'exit':
        print("Goodbye!")
        break
    
    chat_interface.chat(user_input)

Notebook에선 대화형이 인터페이스가 안되는 듯..?
